In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic(base_url="https://www.xiaoyaoapi.com")
model = "claude-sonnet-5"

In [5]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None):
    params = {"model": model, "max_tokens": 1000, "messages": messages}

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return "".join([block.text for block in message.content if block.type == "text"])

In [6]:
import json


def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {"output": output, "test_case": test_case, "score": score}


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results


with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [7]:
print(json.dumps(results, indent=2))

[
  {
    "output": "```python\nimport re\n\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validate an AWS S3 bucket name against S3 bucket naming rules.\n\n    Rules enforced:\n    - Length must be between 3 and 63 characters.\n    - Only lowercase letters, numbers, hyphens, and periods are allowed.\n    - Must start and end with a letter or number.\n    - Must not contain two adjacent periods.\n    - Must not be formatted as an IP address (e.g., 192.168.5.4).\n\n    Args:\n        bucket_name: The S3 bucket name to validate.\n\n    Returns:\n        True if the bucket name is valid, False otherwise.\n    \"\"\"\n    if not isinstance(bucket_name, str):\n        return False\n\n    if not (3 <= len(bucket_name) <= 63):\n        return False\n\n    # Only lowercase letters, digits, hyphens, and periods allowed.\n    if not re.fullmatch(r\"[a-z0-9.\\-]+\", bucket_name):\n        return False\n\n    # Must start and end with a letter or number.\n    if not re.